<!--
 Licensed to the Apache Software Foundation (ASF) under one
 or more contributor license agreements.  See the NOTICE file
 distributed with this work for additional information
 regarding copyright ownership.  The ASF licenses this file
 to you under the Apache License, Version 2.0 (the
 "License"); you may not use this file except in compliance
 with the License.  You may obtain a copy of the License at

   http://www.apache.org/licenses/LICENSE-2.0

 Unless required by applicable law or agreed to in writing,
 software distributed under the License is distributed on an
 "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
 KIND, either express or implied.  See the License for the
 specific language governing permissions and limitations
 under the License.
-->

# Did this farmland green up?

A complete raster pipeline. We answer:

> **Between two satellite scenes a season apart, which farm parcels in this AOI greened up the most?**

End-to-end on Sedona's raster surface: load two GeoTIFFs through the new tiling raster data source, compute NDVI per scene with `RS_MapAlgebra`, take their difference for a "greening" delta raster, run `RS_ZonalStats` per parcel, rank, clip the delta raster to the top parcel with `RS_Clip`, round-trip through `RS_AsCOG` to prove the result is a valid Cloud-Optimized GeoTIFF, and visualize the four stages as a single matplotlib figure.

The two source scenes are **synthesized** at the top of the notebook (256 × 256 px, 2 bands each: red + NIR, EPSG:4326) so the notebook is fully reproducible and ships no new bytes. The same code path runs unchanged against real Sentinel-2 chips — only the raster filenames change.

**Requires Sedona ≥ 1.9.0.** The `raster` data source (auto-tiling GeoTIFF reader, GH-2672) and `RS_AsCOG` (GH-2652) land in 1.9.0; the notebook will fail on older versions of the docker image.

## 1. Connect to Spark through SedonaContext

In [ ]:
from sedona.spark import SedonaContext

config = SedonaContext.builder().master("spark://localhost:7077").getOrCreate()
sedona = SedonaContext.create(config)

## 2. Synthesize two scenes a season apart

We build two 256×256, 2-band GeoTIFFs in `/tmp` representing red and near-infrared reflectance over the same AOI. The "before" scene is mostly bare; the "after" scene has a circular field of vegetation in the south-west corner with elevated NIR. Pixel values are scaled to the same 0-10000 reflectance range as Sentinel-2 surface-reflectance products so the NDVI math is identical to the real-world case.

In [ ]:
import os

import numpy as np
import rasterio
from rasterio.transform import from_bounds

WORK = "/tmp/veg-change"
os.makedirs(WORK, exist_ok=True)

AOI = (-91.10, 41.50, -91.00, 41.60)  # (xmin, ymin, xmax, ymax) in EPSG:4326
W = H = 256
transform = from_bounds(*AOI, W, H)
rng = np.random.default_rng(42)

ys, xs = np.mgrid[0:H, 0:W]
field_mask = ((xs - 64) ** 2 + (ys - 192) ** 2) < 50**2  # circular field


def synth_scene(green_strength):
    red = (1500 + 200 * rng.standard_normal((H, W))).clip(0, 10000)
    nir = (1800 + 200 * rng.standard_normal((H, W))).clip(0, 10000)
    nir = np.where(field_mask, nir + green_strength, nir)
    return red.astype("uint16"), nir.astype("uint16")


for name, strength in [("before", 200), ("after", 4000)]:
    red, nir = synth_scene(strength)
    with rasterio.open(
        f"{WORK}/scene_{name}.tif",
        "w",
        driver="GTiff",
        tiled=True,
        blockxsize=256,
        blockysize=256,
        height=H,
        width=W,
        count=2,
        dtype="uint16",
        crs="EPSG:4326",
        transform=transform,
    ) as dst:
        dst.write(red, 1)
        dst.write(nir, 2)
        dst.set_band_description(1, "red")
        dst.set_band_description(2, "nir")

for f in (f"{WORK}/scene_before.tif", f"{WORK}/scene_after.tif"):
    print(f"{f}: {os.path.getsize(f)} bytes")

## 3. Load both scenes with the `raster` data source

`sedona.read.format("raster")` (new in 1.9) auto-tiles GeoTIFFs and yields one `Raster`-typed row per tile, sidestepping Spark's 2 GB record-size ceiling on large GeoTIFFs. Our scenes are tiny so each yields a single tile, but the call shape is identical for multi-gigabyte inputs.

In [ ]:
scenes = (
    sedona.read.format("raster")
    .load(f"{WORK}/scene_*.tif")
    .selectExpr("split(name, '\\\\.')[0] as scene", "rast")
)
scenes.cache()
scenes.show(truncate=80)

## 4. Compute NDVI per scene with `RS_MapAlgebra`

`RS_MapAlgebra(raster, pixelType, script)` runs a single-raster pixel script. The script syntax (`out[0] = ...; rast[i]` indexing) is documented [here](../api/sql/Raster-map-algebra.md). We coerce the result to `D` (double) so the negative side of NDVI survives.

In [ ]:
scenes.createOrReplaceTempView("scenes")

ndvi = sedona.sql("""
    SELECT scene,
           RS_MapAlgebra(
               rast, 'D',
               'out[0] = (rast[1] - rast[0]) / (rast[1] + rast[0] + 0.000001);'
           ) AS rast
    FROM scenes
""")
ndvi.cache()
ndvi.createOrReplaceTempView("ndvi")
ndvi.selectExpr("scene", "RS_BandPixelType(rast, 1) as dtype").show()

## 5. Compute the greening delta raster

The two-raster form `RS_MapAlgebra(rast0, rast1, pixelType, script, noDataValue)` lets us subtract the before-NDVI from the after-NDVI in a single pass. The result is a per-pixel ΔNDVI layer where positive values mean "this pixel got greener".

In [ ]:
delta = sedona.sql("""
    SELECT RS_MapAlgebra(
               (SELECT rast FROM ndvi WHERE scene = 'scene_after'),
               (SELECT rast FROM ndvi WHERE scene = 'scene_before'),
               'D',
               'out[0] = rast0[0] - rast1[0];',
               -9999.0
           ) AS rast
""")
delta.cache()
delta.createOrReplaceTempView("delta")
delta.selectExpr(
    "RS_BandPixelType(rast, 1) as dtype",
    "RS_Width(rast) as w",
    "RS_Height(rast) as h",
).show()

## 6. Synthesize parcel polygons and run `RS_ZonalStats`

We build a 4 × 4 grid of square parcels covering the AOI and compute the mean ΔNDVI per parcel. `RS_ZonalStats(raster, zone, statType)` is the canonical raster→vector aggregation: every pixel inside `zone` contributes to the statistic. Parcels are then ranked by mean ΔNDVI.

In [ ]:
from pyspark.sql import Row

xmin, ymin, xmax, ymax = AOI
step_x = (xmax - xmin) / 4
step_y = (ymax - ymin) / 4
parcel_rows = []
for ix in range(4):
    for iy in range(4):
        x0, x1 = xmin + ix * step_x, xmin + (ix + 1) * step_x
        y0, y1 = ymin + iy * step_y, ymin + (iy + 1) * step_y
        wkt = f"POLYGON(({x0} {y0}, {x1} {y0}, {x1} {y1}, {x0} {y1}, {x0} {y0}))"
        parcel_rows.append(Row(parcel_id=f"P{ix}{iy}", wkt=wkt))

parcels = sedona.createDataFrame(parcel_rows).selectExpr(
    "parcel_id", "ST_GeomFromText(wkt) as geom"
)
parcels.createOrReplaceTempView("parcels")

ranked = sedona.sql("""
    SELECT p.parcel_id,
           ROUND(RS_ZonalStats(d.rast, p.geom, 'mean'), 4) AS mean_delta_ndvi
    FROM parcels p, delta d
    ORDER BY mean_delta_ndvi DESC
""")
ranked.show()

## 7. Clip the delta raster to the top-greening parcel

`RS_Clip(raster, band, geom)` crops the raster to the polygon's extent. We pick the parcel with the highest mean ΔNDVI and zoom in on its delta values.

In [ ]:
top_id = ranked.first()["parcel_id"]
print(f"top-greening parcel: {top_id}")

clipped = sedona.sql(f"""
    SELECT RS_Clip(d.rast, 1, p.geom) AS rast
    FROM delta d, parcels p
    WHERE p.parcel_id = '{top_id}'
""")
clipped.cache()
clipped.createOrReplaceTempView("clipped")
clipped.selectExpr(
    "RS_Width(rast) as w",
    "RS_Height(rast) as h",
).show()

## 8. Round-trip through `RS_AsCOG`

`RS_AsCOG` returns the raster as a binary Cloud-Optimized GeoTIFF byte array. We persist it to disk, then read it back with the same `raster` data source we used to load the input — proving the output is a valid GeoTIFF that downstream tools can stream from object storage.

In [ ]:
cog_bytes = clipped.selectExpr("RS_AsCOG(rast) as cog").first()["cog"]
with open(f"{WORK}/delta_topparcel_cog.tif", "wb") as fh:
    fh.write(cog_bytes)
print(f"COG size: {len(cog_bytes):,} bytes")

roundtrip = sedona.read.format("raster").load(f"{WORK}/delta_topparcel_cog.tif")
roundtrip.selectExpr(
    "RS_Width(rast) as w",
    "RS_Height(rast) as h",
    "RS_BandPixelType(rast, 1) as dtype",
).show()

## 9. Visualize the pipeline as a 4-panel figure

Read the rasters back via `rasterio` and lay them out side by side: NDVI before, NDVI after, ΔNDVI across the full AOI with parcel boundaries overlaid, and the top-parcel close-up.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt


def ndvi_arr(path):
    with rasterio.open(path) as ds:
        red = ds.read(1).astype("float32")
        nir = ds.read(2).astype("float32")
        return (nir - red) / (nir + red + 1e-6)


ndvi_before = ndvi_arr(f"{WORK}/scene_before.tif")
ndvi_after = ndvi_arr(f"{WORK}/scene_after.tif")
delta_arr = ndvi_after - ndvi_before
with rasterio.open(f"{WORK}/delta_topparcel_cog.tif") as ds:
    top_arr = ds.read(1)
    top_extent = (ds.bounds.left, ds.bounds.right, ds.bounds.bottom, ds.bounds.top)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
extent = (AOI[0], AOI[2], AOI[1], AOI[3])
axes[0].imshow(ndvi_before, vmin=-0.2, vmax=0.8, cmap="RdYlGn", extent=extent)
axes[0].set_title("NDVI before")
axes[1].imshow(ndvi_after, vmin=-0.2, vmax=0.8, cmap="RdYlGn", extent=extent)
axes[1].set_title("NDVI after")
axes[2].imshow(delta_arr, vmin=-0.5, vmax=0.5, cmap="PiYG", extent=extent)
axes[2].set_title("ΔNDVI (after − before)")
axes[3].imshow(top_arr, vmin=-0.5, vmax=0.5, cmap="PiYG", extent=top_extent)
axes[3].set_title(f"Top parcel ({top_id}) ΔNDVI")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
fig

## What just happened?

We turned two synthetic Sentinel-2-style scenes into a ranked parcel-level greening report in nine steps:

1. Synthesized two 256×256 red+NIR GeoTIFFs in `/tmp`.
2. Loaded both with `sedona.read.format("raster")` — the new auto-tiling reader handles GeoTIFFs of any size with the same call shape.
3. Computed per-pixel NDVI per scene with single-raster `RS_MapAlgebra`.
4. Subtracted the two NDVI rasters with two-raster `RS_MapAlgebra` to get a ΔNDVI "greening" layer.
5. Synthesized a 4×4 parcel grid and ranked parcels by mean ΔNDVI via `RS_ZonalStats`.
6. Clipped the delta raster to the top-greening parcel with `RS_Clip`.
7. Encoded the result as a Cloud-Optimized GeoTIFF with `RS_AsCOG` and read it back with the `raster` data source — proving the output is valid for cloud-hosted streaming.
8. Plotted the four pipeline stages side by side.

Swap `/tmp/scene_*.tif` for a glob over real Sentinel-2 chips and the same SQL runs on a cluster against terabytes of imagery.